In [42]:
import pandas as pd
import os

## Configuration
Remplacez par le chemin vers les fichiers à traiter

In [49]:
data_path = './input/OSO_TSG_001.csv'
instrument_name = 'OSO_TSG_001' # Suivre le format du numéro de série (owner_type_numéro)
output_dir = "output"
csv_encoding = 'utf-8' # or 'latin-1' depending on your file encoding

## Étape 1 : Charger les données CSV

In [56]:
df = pd.read_csv(
    data_path,
    low_memory=False,
    delimiter=';',
    encoding=csv_encoding,
    usecols=lambda col: not str(col).strip().lower().startswith('unnamed')
)

# Supprime les colonnes entièrement vides
df = df.dropna(axis=1, how='all')

print(df.head())

         Lat       Lng        Gps Time      Date      Time  Temp_int(C)  \
0  47.545515 -2.894622  20825 14152100  25/08/02  14:15:00       31.000   
1  47.545508 -2.894589  20825 14203900  25/08/02  14:21:00       30.875   
2  47.545505 -2.894599  20825 14255700  25/08/02  14:26:00       30.687   
3  47.545501 -2.894587  20825 14311500  25/08/02  14:31:00       30.375   
4  47.545485 -2.894623  20825 14363400  25/08/02  14:37:00       30.750   

   T-SeaLL  P-SeaLL  EC_SeaLL  EC_20_SeaLL  EC_25_SeaLL  Sal_SeaLL  \
0    20.07   1081.8   48045.0      47974.0      53040.0      34.99   
1    20.11   1081.6   49242.0      49128.0      54315.0      35.92   
2    20.10   1081.3   49115.0      49012.0      54187.0      35.84   
3    20.13   1081.9   49347.0      49212.0      54408.0      35.98   
4    20.15   1081.8   49318.0      49162.0      54353.0      35.95   

   P_calc_SeaLL  freq(Hz)  R9|ycell  R8|ycell  R9|R3  R8|R4  
0          0.70    4500.0         1     18892   3414   3419  
1   

## Étape 2 : Adapter au format de la plateforme de donnée

In [57]:
# Fusionner Date et Time en une colonne datetime Python
if {'Date', 'Time'}.issubset(df.columns):
    df['datetime'] = pd.to_datetime(
        df['Date'].astype(str).str.strip() + ' ' + df['Time'].astype(str).str.strip(),
        errors='coerce',
        format='%y/%m/%d %H:%M:%S'
    )
    df = df.drop(columns=['Date', 'Time'], errors='ignore')

# Supprimer les colonnes inutiles
df = df.drop(
    columns=[
        'Gps Time',
        'Pression_ext(hpa)', # Null dans ce dataset, à vérifier pour les autres
        'Temp_air(C)',  # Null dans ce dataset, à vérifier pour les autres
        'P_calc_SeaLL',
        'freq(Hz)',
        'R9|ycell',
        'R8|ycell',
        'R9|R3',
        'R8|R4',
        'Temp_int(C)' # Peut être utile pour les analyses futures, à vérifier
    ],
    errors='ignore'
)

# Renommer les colonnes pour correspondre au format de la plateforme
df = df.rename(columns={
    'Lat': 'lat',
    'Lng': 'long',         
    'T-SeaLL': 'sea_temp',
    'P-SeaLL': 'sea_pres',
    'EC_SeaLL': 'ec_abs',
    'EC_20_SeaLL': 'ec20',
    'EC_25_SeaLL': 'ec25',
    'Sal_SeaLL': 'sea_sal'
})



print(df.head())

         lat      long  sea_temp  sea_pres   ec_abs     ec20     ec25  \
0  47.545515 -2.894622     20.07    1081.8  48045.0  47974.0  53040.0   
1  47.545508 -2.894589     20.11    1081.6  49242.0  49128.0  54315.0   
2  47.545505 -2.894599     20.10    1081.3  49115.0  49012.0  54187.0   
3  47.545501 -2.894587     20.13    1081.9  49347.0  49212.0  54408.0   
4  47.545485 -2.894623     20.15    1081.8  49318.0  49162.0  54353.0   

   sea_sal            datetime  
0    34.99 2025-08-02 14:15:00  
1    35.92 2025-08-02 14:21:00  
2    35.84 2025-08-02 14:26:00  
3    35.98 2025-08-02 14:31:00  
4    35.95 2025-08-02 14:37:00  


## Étape 3 : Exporter le résultat

In [58]:
os.makedirs(output_dir, exist_ok=True)

if not df.empty:
    start_at = df['datetime'].min().strftime('%Y-%m-%d_%H_%M_%S')
    output_csv_final = os.path.join(output_dir, f"{instrument_name}_{start_at}.csv")
    df.to_csv(output_csv_final, sep=';', date_format='%Y-%m-%dT%H:%M:%SZ', index=False)
    print(f"Fichier exporté : {output_csv_final}")
else:
    print("Aucune donnée interpolée à exporter.")

Fichier exporté : output/OSO_TSG_001_2025-08-02_14_15_00.csv
